In [ ]:
from pathlib import Path
import sys, re, html, importlib.util
import importlib.util
if importlib.util.find_spec('duckdb') is None:
    raise ModuleNotFoundError('duckdb is missing from this Jupyter kernel; run %pip install -r requirements.txt, then restart the kernel.')
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'ypipe').is_dir(): PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from ypipe import config
from ypipe.nlp import STOP

SEED = 42
np.random.seed(SEED)
MIN_TITLES = config.MIN_TITLES_FOR_TOPICS  # dùng ngưỡng pipeline để tránh cụm thiếu dữ liệu
TOPIC_CANDIDATES = tuple(sorted({max(2, config.N_TOPICS-d) for d in (6,4,2,0)}))
MIN_DF = 3
MAX_DF = 0.8
MAX_FEATURES = 30000
SILHOUETTE_SAMPLE = 5000  # giới hạn chi phí tính silhouette trên tập lớn
MIN_CLUSTER_VIDEOS = 10  # chỉ mô tả cụm có ít nhất 10 video
LOW_PURITY_THRESHOLD = 0.5  # ngưỡng chọn cụm title không khớp rõ một thể loại API
SEGMENTATION_SAMPLE = 2000  # giới hạn thời gian cho tokenizer tiếng Việt tùy chọn
config.DATA.mkdir(parents=True, exist_ok=True)
_previous_con = globals().get('con')
if _previous_con is not None:
    try:
        _previous_con.close()
    except Exception:
        pass
class _NotebookQueryResult:
    def __init__(self, frame):
        self._frame = frame
    def df(self):
        return self._frame.copy()
    def fetchdf(self):
        return self._frame.copy()
    def fetchone(self):
        if self._frame.empty:
            return None
        return tuple(self._frame.iloc[0].tolist())
    def fetchall(self):
        return list(self._frame.itertuples(index=False, name=None))

class _NotebookReadOnlyDB:
    """Run each read query on a short-lived connection to avoid retaining a file lock."""
    def __init__(self, path):
        self.path = Path(path)
    def execute(self, sql, parameters=None):
        if not self.path.exists():
            if sql.strip().lower() == 'show tables':
                return _NotebookQueryResult(pd.DataFrame({'name': pd.Series(dtype='object')}))
            return _NotebookQueryResult(pd.DataFrame())
        db = duckdb.connect(str(self.path), read_only=True)
        try:
            result = db.execute(sql) if parameters is None else db.execute(sql, parameters)
            frame = result.fetchdf()
        finally:
            db.close()
        return _NotebookQueryResult(frame)
    def close(self):
        # Queries already close their own connection before returning.
        return None

con = _NotebookReadOnlyDB(config.DB_PATH)
TABLES = set(con.execute('SHOW TABLES').fetchdf()['name'])
print(f'Database: {config.DB_PATH} | seed={SEED} | N_TOPICS cấu hình={config.N_TOPICS}')

# Chủ đề từ tiêu đề — Stage 5

## Câu hỏi
Các tiêu đề video tạo thành những cụm từ vựng nào, và các cụm này liên hệ ra sao với thể loại do YouTube API cung cấp?

## Phương pháp
Làm sạch HTML/URL và khoảng trắng, vector hóa bằng TF-IDF unigram/bigram rồi chạy MiniBatchKMeans với nhiều giá trị k suy ra từ `config.N_TOPICS`. Chọn k có silhouette cao nhất trong các giá trị hợp lệ. `category_name` chỉ là **nhãn yếu** để đối chiếu, không phải nhãn chuẩn. `video_topics.topic_id` có thể đổi khi huấn luyện lại; dùng `topics.label` để đọc nhãn đã lưu trong pipeline.

In [ ]:
raw = con.execute('SELECT video_id,title FROM watch_raw').df() if 'watch_raw' in TABLES else pd.DataFrame(columns=['video_id','title'])
facts = con.execute('SELECT video_id,category_name,metadata_status FROM video_facts').df() if 'video_facts' in TABLES else pd.DataFrame(columns=['video_id','category_name','metadata_status'])
stored_map = con.execute('SELECT video_id,topic_id FROM video_topics').df() if 'video_topics' in TABLES else pd.DataFrame(columns=['video_id','topic_id'])
stored_labels = con.execute('SELECT topic_id,label,n_videos FROM topics').df() if 'topics' in TABLES else pd.DataFrame(columns=['topic_id','label','n_videos'])

def clean_title(value):
    text = html.unescape(str(value or ''))
    text = re.sub(r'https?://\S+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

titles = raw.copy()
if len(titles):
    titles['title'] = titles['title'].fillna('').map(clean_title)
    titles = titles[titles['title'].str.len()>0].drop_duplicates('video_id').reset_index(drop=True)
titles = titles.merge(facts[['video_id','category_name','metadata_status']],on='video_id',how='left')
titles['category_name'] = titles['category_name'].fillna('Chưa phân loại')
print(f'Tiêu đề không rỗng, video duy nhất: {len(titles):,}; ngưỡng tối thiểu: {MIN_TITLES:,}')

k_scores = []
clustered = pd.DataFrame(columns=['video_id','title','category_name','exploratory_topic_id'])
topic_terms = pd.DataFrame(columns=['exploratory_topic_id','label','n_videos'])
best_k = None; best_silhouette = np.nan; best_model = None; vectorizer = None; X = None
if len(titles) >= MIN_TITLES:
    try:
        vectorizer = TfidfVectorizer(lowercase=True,ngram_range=(1,2),min_df=MIN_DF,max_df=MAX_DF,
            max_features=MAX_FEATURES,token_pattern=r'(?u)\b[^\W\d_]{2,}\b',stop_words=list(STOP))
        X = vectorizer.fit_transform(titles['title'])
        valid_ks = [k for k in TOPIC_CANDIDATES if 2 <= k < len(titles) and X.shape[1] >= k]
        for k in valid_ks:
            model = MiniBatchKMeans(n_clusters=k,random_state=SEED,n_init=5,batch_size=4096).fit(X)
            sil = silhouette_score(X,model.labels_,sample_size=min(SILHOUETTE_SAMPLE,len(titles)),random_state=SEED)
            k_scores.append({'k':k,'silhouette':sil})
        if k_scores:
            score_table = pd.DataFrame(k_scores).sort_values('silhouette',ascending=False)
            best_k = int(score_table.iloc[0]['k']); best_silhouette = float(score_table.iloc[0]['silhouette'])
            best_model = MiniBatchKMeans(n_clusters=best_k,random_state=SEED,n_init=10,batch_size=4096).fit(X)
            terms = np.asarray(vectorizer.get_feature_names_out())
            labels=[]
            for topic_id,center in enumerate(best_model.cluster_centers_):
                top_terms=terms[np.argsort(center)[::-1][:6]]
                labels.append({'exploratory_topic_id':topic_id,'label':', '.join(top_terms),
                               'n_videos':int((best_model.labels_==topic_id).sum())})
            topic_terms=pd.DataFrame(labels)
            clustered=titles[['video_id','title','category_name']].copy()
            clustered['exploratory_topic_id']=best_model.labels_
        else:
            print('Không có k hợp lệ sau khi kiểm tra số title và số chiều TF-IDF.')
    except ValueError as exc:
        print(f'Không thể vector hóa title với cấu hình hiện tại: {exc}')
else:
    print('Chưa đủ title theo MIN_TITLES; bỏ qua topic modeling trong notebook.')

if len(stored_map) and len(stored_labels):
    stored_readable=stored_map.merge(stored_labels[['topic_id','label','n_videos']],on='topic_id',how='left')
    print('Topic đã lưu trong pipeline (đọc tên qua topics.label):')
    display(stored_readable.groupby(['topic_id','label'],dropna=False).agg(video_count=('video_id','nunique'),n_videos=('n_videos','first')).reset_index())
else:
    print('Chưa có bảng topic đã lưu; notebook vẫn có thể fit topic khám phá riêng nếu đủ title.')

## Kết quả: chọn k và nhãn từ khóa
Silhouette được tính trên TF-IDF thưa; phép chọn k là heuristic và không chứng minh cấu trúc chủ đề tự nhiên. Nhãn cụm dưới đây là các từ có trọng số cao, không phải taxonomy chính thức.

In [ ]:
if k_scores:
    score_table=pd.DataFrame(k_scores).sort_values('k')
    fig,ax=plt.subplots(figsize=(7,3.5)); ax.plot(score_table.k,score_table.silhouette,marker='o',color='#2563eb')
    ax.set_title('Silhouette theo số cụm k'); ax.set_xlabel('Số cụm k'); ax.set_ylabel('Silhouette (không đơn vị)')
    ax.set_xticks(score_table.k); plt.tight_layout(); plt.show()
    print(f'Chọn k={best_k}, silhouette={best_silhouette:.3f}')
    display(topic_terms.rename(columns={'exploratory_topic_id':'Mã cụm khám phá','label':'Từ khóa đại diện','n_videos':'Số video'}))
else:
    print('Chưa có kết quả chọn k để hiển thị.')

## Kết quả: đối chiếu với thể loại API (nhãn yếu)
Bảng chéo, purity, NMI và ARI chỉ đo mức độ liên hệ giữa cụm title không giám sát với `category_name`. Thể loại API không phải nhãn chuẩn cho nội dung, nên các chỉ số này không phải accuracy hay chất lượng phân loại có giám sát.

In [ ]:
weak = clustered[(clustered['category_name']!='Chưa phân loại')].copy() if len(clustered) else pd.DataFrame()
if len(weak) and weak['category_name'].nunique()>1 and weak['exploratory_topic_id'].nunique()>1:
    cross=pd.crosstab(weak['exploratory_topic_id'],weak['category_name'])
    cluster_sizes=cross.sum(axis=1)
    purity=float(cross.max(axis=1).sum()/cross.to_numpy().sum())
    nmi=normalized_mutual_info_score(weak['category_name'],weak['exploratory_topic_id'])
    ari=adjusted_rand_score(weak['category_name'],weak['exploratory_topic_id'])
    print(f'Purity={purity:.3f} | NMI={nmi:.3f} | ARI={ari:.3f} | video có category={len(weak):,}')
    display(cross)
    topic_category=cross.idxmax(axis=1).rename('dominant_category').to_frame()
    topic_category['purity']=cross.max(axis=1)/cluster_sizes
    topic_category['n_videos']=cluster_sizes
    mismatched=topic_category[(topic_category.purity<LOW_PURITY_THRESHOLD)&(topic_category.n_videos>=MIN_CLUSTER_VIDEOS)]
    if len(mismatched):
        examples=(weak[weak.exploratory_topic_id.isin(mismatched.index)]
                 .merge(topic_category,left_on='exploratory_topic_id',right_index=True)
                 .query('category_name != dominant_category')
                 .groupby('exploratory_topic_id')['title'].apply(lambda s: list(s.head(5))).to_frame())
        display(mismatched.join(examples).rename(columns={'dominant_category':'Thể loại trội','purity':'Purity','n_videos':'Số video','title':'Ví dụ title lệch thể loại'}))
    else:
        print('Không có cụm đủ lớn dưới LOW_PURITY_THRESHOLD; điều này không chứng minh phân loại đúng.')
else:
    print('Chưa đủ title có category để tính đối chiếu nhãn yếu.')

## So sánh tách từ tiếng Việt (tùy chọn)
Nếu `underthesea` hoặc `pyvi` đã có sẵn, cell này so sánh TF-IDF baseline với tokenizer trên một mẫu giới hạn; không tự cài package hay tải embedding.

In [ ]:
segmenter_name=None; segmenter=None
if importlib.util.find_spec('underthesea'):
    from underthesea import word_tokenize
    segmenter=lambda text: word_tokenize(text,format='text')
    segmenter_name='underthesea'
elif importlib.util.find_spec('pyvi'):
    from pyvi import ViTokenizer
    segmenter=ViTokenizer.tokenize
    segmenter_name='pyvi'

if segmenter is not None and best_k is not None:
    segment_sample=titles.sample(min(SEGMENTATION_SAMPLE,len(titles)),random_state=SEED).copy()
    try:
        segmented=segment_sample['title'].map(segmenter)
        seg_vec=TfidfVectorizer(lowercase=True,ngram_range=(1,2),min_df=2,max_df=MAX_DF,
            max_features=MAX_FEATURES,token_pattern=r'(?u)\b[^\W\d_]{2,}(?:_[^\W\d_]{2,})*\b',stop_words=list(STOP))
        X_seg=seg_vec.fit_transform(segmented)
        k_seg=min(best_k,len(segment_sample)-1)
        if k_seg>=2 and X_seg.shape[1]>=k_seg:
            seg_model=MiniBatchKMeans(n_clusters=k_seg,random_state=SEED,n_init=5,batch_size=2048).fit(X_seg)
            seg_sil=silhouette_score(X_seg,seg_model.labels_,sample_size=min(SILHOUETTE_SAMPLE,len(segment_sample)),random_state=SEED)
            base_vec=TfidfVectorizer(lowercase=True,ngram_range=(1,2),min_df=2,max_df=MAX_DF,max_features=MAX_FEATURES,
                token_pattern=r'(?u)\b[^\W\d_]{2,}\b',stop_words=list(STOP))
            X_base=base_vec.fit_transform(segment_sample['title'])
            base_model=MiniBatchKMeans(n_clusters=k_seg,random_state=SEED,n_init=5,batch_size=2048).fit(X_base)
            base_sil=silhouette_score(X_base,base_model.labels_,sample_size=min(SILHOUETTE_SAMPLE,len(segment_sample)),random_state=SEED)
            display(pd.DataFrame([{'Cách vector hóa':'TF-IDF baseline','Silhouette':base_sil},
                                  {'Cách vector hóa':f'TF-IDF + {segmenter_name}','Silhouette':seg_sil}]))
        else: print('Mẫu tokenizer chưa đủ title/đặc trưng để so sánh.')
    except ValueError as exc: print(f'Không thể so sánh tokenizer tùy chọn: {exc}')
else:
    print('Không có underthesea/pyvi hoặc chưa fit được baseline; bỏ qua so sánh tách từ tùy chọn.')

## So sánh embedding đa ngôn ngữ (tùy chọn)
Nếu `YTPIPE_TOPIC_EMBEDDING_MODEL` được cấu hình, cell sau so sánh silhouette trên một mẫu title với baseline TF-IDF. Việc tải/chọn model do người chạy chủ động cấu hình.

In [ ]:
if config.TOPIC_EMBEDDING_MODEL and best_k is not None:
    try:
        from sentence_transformers import SentenceTransformer
        embed_sample=titles.sample(min(SEGMENTATION_SAMPLE,len(titles)),random_state=SEED).copy()
        embed_text=embed_sample['title'].fillna('').astype(str)
        encoder=SentenceTransformer(config.TOPIC_EMBEDDING_MODEL)
        vectors=encoder.encode(embed_text.tolist(),batch_size=config.TOPIC_EMBEDDING_BATCH_SIZE,
            show_progress_bar=False,convert_to_numpy=True,normalize_embeddings=True)
        k_embed=min(best_k,len(embed_sample)-1)
        embed_clusters=MiniBatchKMeans(n_clusters=k_embed,random_state=SEED,n_init=5,batch_size=2048).fit_predict(vectors)
        embed_sil=silhouette_score(vectors,embed_clusters,sample_size=min(SILHOUETTE_SAMPLE,len(embed_sample)),random_state=SEED)
        baseline_vec=TfidfVectorizer(lowercase=True,ngram_range=(1,2),min_df=MIN_DF,max_df=MAX_DF,
            max_features=MAX_FEATURES,token_pattern=r'(?u)\b[^\W\d_]{2,}\b',stop_words=list(STOP))
        baseline_X=baseline_vec.fit_transform(embed_text)
        baseline_clusters=MiniBatchKMeans(n_clusters=k_embed,random_state=SEED,n_init=5,batch_size=2048).fit_predict(baseline_X)
        baseline_sil=silhouette_score(baseline_X,baseline_clusters,sample_size=min(SILHOUETTE_SAMPLE,len(embed_sample)),random_state=SEED)
        display(pd.DataFrame([{'Cách vector hóa':'TF-IDF baseline','Silhouette':baseline_sil},
                              {'Cách vector hóa':f'Embedding: {config.TOPIC_EMBEDDING_MODEL}','Silhouette':embed_sil}]))
    except ImportError:
        print('Đã cấu hình embedding nhưng sentence-transformers chưa được cài.')
else:
    print('Embedding không bật; giữ nguyên TF-IDF baseline.')

## Insight
Đọc cụm theo ví dụ title và từ khóa, rồi dùng bảng chéo để tìm cụm không trùng khớp với thể loại API. Không đặt tên cụm như thể đó là nhãn chuẩn; `exploratory_topic_id` trong lần fit notebook độc lập với `video_topics.topic_id` đã lưu từ pipeline.

## Hạn chế
Tokenization baseline chia theo âm tiết/khoảng trắng nên cụm từ tiếng Việt chưa được tách từ nếu tokenizer tùy chọn không cài. KMeans phụ thuộc k và seed; title ngắn/nhiễu làm giảm chất lượng. `category_name` chỉ là nhãn yếu; purity/NMI/ARI không phải accuracy. Mẫu chỉ gồm video người tham gia đã xem và có title.

In [ ]:
# Đóng connection để pipeline notebook có thể ghi DuckDB sau khi chạy notebook.
if 'con' in globals() and con is not None:
    con.close()
    print('Đã đóng connection DuckDB; có thể chạy pipeline.')